In [ ]:
# PREPARANDO DADOS DO DATAFRAME TITANIC PARA K-NN

Para preparar a base do Titanic para um algoritmo baseado em distâncias como o k-NN, o foco total deve ser transformar tudo em números e garantir que esses números estejam na mesma escala. O k-NN calcula a distância entre linhas (registros) como se fossem pontos em um gráfico de várias dimensões.

Aqui estão as 5 etapas fundamentais utilizando o Pandas:

1. Seleção de Atributos (Feature Selection)
Nem toda coluna ajuda o k-NN. Colunas como PassengerId, Name e Ticket são identificadores únicos ou strings complexas que não possuem uma "distância" matemática útil para a sobrevivência.

Ação: Remover colunas irrelevantes.

Python
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
2. Tratamento de Valores Ausentes (Imputação)
O k-NN não consegue calcular a distância se houver um NaN (Not a Number). Na base do Titanic, a coluna Age costuma ter muitos nulos.

Ação: Preencher a idade com a mediana (mais robusta a outliers) e o porto de embarque (Embarked) com a moda.

Python
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
3. Codificação de Variáveis Categóricas (Encoding)
O k-NN não sabe a distância entre "Male" e "Female". Precisamos converter categorias em números.

One-Hot Encoding: Para Embarked (C, Q, S), criamos colunas binárias.

Label Encoding: Para Sex, podemos transformar em 0 e 1.

Python
# Transformando Sexo em 0 e 1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Criando dummies para Embarked (C, Q, S viram 3 colunas)
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
4. Engenharia de Atributos (Opcional, mas Recomendado)
Podemos simplificar colunas para ajudar o modelo. Por exemplo, somar SibSp (irmãos/cônjuges) e Parch (pais/filhos) para criar o tamanho da família.

Ação: Criar a coluna FamilySize.

Python
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df = df.drop(['SibSp', 'Parch'], axis=1)
5. Escalonamento de Atributos (Feature Scaling) - CRUCIAL
Esta é a etapa mais importante para o k-NN. A coluna Fare (Tarifa) pode variar de 0 a 500, enquanto Pclass varia de 1 a 3. Se não escalonarmos, a Tarifa terá um peso muito maior na distância do que a Classe, "atropelando" o modelo.

Ação: Aplicar Normalização (0 a 1) ou Padronização.

Python
# Exemplo de Normalização Min-Max no Pandas
def min_max_scaling(column):
    return (column - column.min()) / (column.max() - column.min())

df['Age'] = min_max_scaling(df['Age'])
df['Fare'] = min_max_scaling(df['Fare'])
Resumo do Fluxo para o k-NN:
Limpar: Remover IDs e nomes.

Imputar: Preencher buracos em Age.

Codificar: Sex -> 0/1, Embarked -> Dummies.

Escalonar: Deixar Age, Fare e FamilySize entre 0 e 1.

Após essas etapas, seu DataFrame estará pronto: uma matriz puramente numérica onde cada linha é um vetor de características pronto para ter sua distância calculada em relação aos "vizinhos".

In [3]:
import pandas as pd
import numpy as np

# 1. Carregando os dados (Substitua pelo caminho do seu arquivo)
df = pd.read_csv('/content/sample_data/train.csv')

print("--- Base Original ---")
print(df.head())

--- Base Original ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.05

In [6]:

# 2. Imputação de Nulos (Age)
df['Age'] = df['Age'].fillna(df['Age'].median())
print("--- idades nulas preenchidas ---")
print(df.head())

df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
print("--- PORTOS DE Embarque ---")
print(df.head())


--- idades nulas preenchidas ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            37

In [7]:
# 3. Encoding (Transformando categorias em números)
# Sex: male=0, female=1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
print("--- Sexo substituido - Masc = 0, Fem = 1 ---")
print(df.head())


--- Sexo substituido - Masc = 0, Fem = 1 ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name  Sex   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris    0  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1  38.0      1      0   
2                             Heikkinen, Miss. Laina    1  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1  35.0      1      0   
4                           Allen, Mr. William Henry    0  35.0      0      0   

             Ticket     Fare Cabin Embarked  
0         A/5 21171   7.2500   NaN        S  
1          PC 17599  71.2833   C85        C  
2  STON/O2. 3101282   7.9250   NaN        S  
3            113803  53.1000  C123        S  
4            373450  

In [8]:
# Embarked: Get Dummies (C, Q, S)
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
print("--- Substituição da sigla Embarked por True / False em colunas ---")
print(df.head())



--- Substituição da sigla Embarked por True / False em colunas ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name  Sex   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris    0  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1  38.0      1      0   
2                             Heikkinen, Miss. Laina    1  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1  35.0      1      0   
4                           Allen, Mr. William Henry    0  35.0      0      0   

             Ticket     Fare Cabin  Embarked_Q  Embarked_S  
0         A/5 21171   7.2500   NaN       False        True  
1          PC 17599  71.2833   C85       False       False  
2  STON/O2. 3101282   7.9250   NaN       False

In [9]:
# 4. Escalonamento Min-Max (Manual com Pandas)
# Importante: Fazemos isso para que Fare (0-71) não 'atropele' Age (0-38)
cols_to_scale = ['Age', 'Fare', 'Pclass']
for col in cols_to_scale:
    df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
print("--- Aplicado Escalonamento das Variaveis Fare e Age ---")
print(df.head())


--- Aplicado Escalonamento das Variaveis Fare e Age ---
   PassengerId  Survived  Pclass  \
0            1         0     1.0   
1            2         1     0.0   
2            3         1     1.0   
3            4         1     0.0   
4            5         0     1.0   

                                                Name  Sex       Age  SibSp  \
0                            Braund, Mr. Owen Harris    0  0.271174      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1  0.472229      1   
2                             Heikkinen, Miss. Laina    1  0.321438      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1  0.434531      1   
4                           Allen, Mr. William Henry    0  0.434531      0   

   Parch            Ticket      Fare Cabin  Embarked_Q  Embarked_S  
0      0         A/5 21171  0.014151   NaN       False        True  
1      0          PC 17599  0.139136   C85       False       False  
2      0  STON/O2. 3101282  0.015469   NaN       Fa

In [10]:
# base pronta para o k-NN
print("\n--- Base Pronta para k-NN ---")
print(df.head())


--- Base Pronta para k-NN ---
   PassengerId  Survived  Pclass  \
0            1         0     1.0   
1            2         1     0.0   
2            3         1     1.0   
3            4         1     0.0   
4            5         0     1.0   

                                                Name  Sex       Age  SibSp  \
0                            Braund, Mr. Owen Harris    0  0.271174      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1  0.472229      1   
2                             Heikkinen, Miss. Laina    1  0.321438      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1  0.434531      1   
4                           Allen, Mr. William Henry    0  0.434531      0   

   Parch            Ticket      Fare Cabin  Embarked_Q  Embarked_S  
0      0         A/5 21171  0.014151   NaN       False        True  
1      0          PC 17599  0.139136   C85       False       False  
2      0  STON/O2. 3101282  0.015469   NaN       False        True  
3      